In [1]:
import os
import sys
import importlib
import datetime
import requests
import getpass
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm import tqdm
import pandas as pd
import numpy as np
from garth.exc import GarthException, GarthHTTPError
from garminconnect import (
    Garmin,
    GarminConnectAuthenticationError,
    GarminConnectConnectionError,
    GarminConnectTooManyRequestsError,
)

import lib

importlib.reload(lib)

project = lib.Project()
log = lib.getLogger(project.name)

DATE_FORMAT = '%Y-%m-%d'


# Garmin helper functions to interact with API
Taken from example file provided in documentation to ensure safe usage with API

In [2]:
def safe_api_call(api_method, *args, **kwargs):
    """
    Safe API call wrapper with comprehensive error handling.

    This demonstrates the error handling patterns used throughout the library.
    Returns (success: bool, result: Any, error_message: str)
    """
    try:
        result = api_method(*args, **kwargs)
        return True, result, None

    except GarthHTTPError as e:
        # Handle specific HTTP errors gracefully
        error_str = str(e)
        status_code = getattr(getattr(e, "response", None), "status_code", None)

        if status_code == 400 or "400" in error_str:
            return (
                False,
                None,
                "Endpoint not available (400 Bad Request) - Feature may not be enabled for your account",
            )
        elif status_code == 401 or "401" in error_str:
            return (
                False,
                None,
                "Authentication required (401 Unauthorized) - Please re-authenticate",
            )
        elif status_code == 403 or "403" in error_str:
            return (
                False,
                None,
                "Access denied (403 Forbidden) - Account may not have permission",
            )
        elif status_code == 404 or "404" in error_str:
            return (
                False,
                None,
                "Endpoint not found (404) - Feature may have been moved or removed",
            )
        elif status_code == 429 or "429" in error_str:
            return (
                False,
                None,
                "Rate limit exceeded (429) - Please wait before making more requests",
            )
        elif status_code == 500 or "500" in error_str:
            return (
                False,
                None,
                "Server error (500) - Garmin's servers are experiencing issues",
            )
        elif status_code == 503 or "503" in error_str:
            return (
                False,
                None,
                "Service unavailable (503) - Garmin's servers are temporarily unavailable",
            )
        else:
            return False, None, f"HTTP error: {e}"

    except FileNotFoundError:
        return (
            False,
            None,
            "No valid tokens found. Please login with your email/password to create new tokens.",
        )

    except GarminConnectAuthenticationError as e:
        return False, None, f"Authentication issue: {e}"

    except GarminConnectConnectionError as e:
        return False, None, f"Connection issue: {e}"

    except GarminConnectTooManyRequestsError as e:
        return False, None, f"Rate limit exceeded: {e}"

    except Exception as e:
        return False, None, f"Unexpected error: {e}"

def get_credentials():
    """Get email and password from environment or user input."""
    email = os.getenv("EMAIL")
    password = os.getenv("PASSWORD")

    if not email:
        email = input("Login email: ")
    if not password:
        password = getpass("Enter password: ")

    return email, password

def init_api() -> Garmin | None:
    """Initialize Garmin API with authentication and token management."""

    # Configure token storage
    tokenstore = os.getenv("GARMINTOKENS", "~/.garminconnect")
    tokenstore_path = Path(tokenstore).expanduser()

    print(f"🔐 Token storage: {tokenstore_path}")

    # Check if token files exist
    if tokenstore_path.exists():
        print("📄 Found existing token directory")
        token_files = list(tokenstore_path.glob("*.json"))
        if token_files:
            print(
                f"🔑 Found {len(token_files)} token file(s): {[f.name for f in token_files]}"
            )
        else:
            print("⚠️ Token directory exists but no token files found")
    else:
        print("📭 No existing token directory found")

    # First try to login with stored tokens
    try:
        print("🔄 Attempting to use saved authentication tokens...")
        garmin = Garmin()
        garmin.login(str(tokenstore_path))
        print("✅ Successfully logged in using saved tokens!")
        return garmin

    except (
        FileNotFoundError,
        GarthHTTPError,
        GarminConnectAuthenticationError,
        GarminConnectConnectionError,
    ):
        print("🔑 No valid tokens found. Requesting fresh login credentials.")

    # Loop for credential entry with retry on auth failure
    while True:
        try:
            # Get credentials
            email, password = get_credentials()

            print("� Logging in with credentials...")
            garmin = Garmin(
                email=email, password=password, is_cn=False, return_on_mfa=True
            )
            result1, result2 = garmin.login()

            if result1 == "needs_mfa":
                print("🔐 Multi-factor authentication required")

                mfa_code = input("Please enter your MFA code: ")
                print("🔄 Submitting MFA code...")

                try:
                    garmin.resume_login(result2, mfa_code)
                    print("✅ MFA authentication successful!")

                except GarthHTTPError as garth_error:
                    # Handle specific HTTP errors from MFA
                    error_str = str(garth_error)
                    if "429" in error_str and "Too Many Requests" in error_str:
                        print("❌ Too many MFA attempts")
                        print("💡 Please wait 30 minutes before trying again")
                        sys.exit(1)
                    elif "401" in error_str or "403" in error_str:
                        print("❌ Invalid MFA code")
                        print("💡 Please verify your MFA code and try again")
                        continue
                    else:
                        # Other HTTP errors - don't retry
                        print(f"❌ MFA authentication failed: {garth_error}")
                        sys.exit(1)

                except GarthException as garth_error:
                    print(f"❌ MFA authentication failed: {garth_error}")
                    print("💡 Please verify your MFA code and try again")
                    continue

            # Save tokens for future use
            garmin.garth.dump(str(tokenstore_path))
            print(f"💾 Authentication tokens saved to: {tokenstore_path}")
            print("✅ Login successful!")
            return garmin

        except GarminConnectAuthenticationError:
            print("❌ Authentication failed:")
            print("💡 Please check your username and password and try again")
            # Continue the loop to retry
            continue

        except (
            FileNotFoundError,
            GarthHTTPError,
            GarminConnectConnectionError,
            requests.exceptions.HTTPError,
        ) as err:
            print(f"❌ Connection error: {err}")
            print("💡 Please check your internet connection and try again")
            return None

        except KeyboardInterrupt:
            print("\n👋 Cancelled by user")
            return None


# Get Garmin client

In [4]:
api = init_api()

🔐 Token storage: /Users/christophermagno/.garminconnect
📄 Found existing token directory
🔑 Found 2 token file(s): ['oauth2_token.json', 'oauth1_token.json']
🔄 Attempting to use saved authentication tokens...
✅ Successfully logged in using saved tokens!


# Helper functions for datetime

In [3]:
def _get_date_string(date):
    return date.strftime(DATE_FORMAT)

def today():
    return datetime.date.today().strftime(DATE_FORMAT)


def get_date_range(start=None, rng=None):
    start = start or datetime.datetime.today()
    rng = rng or int(start.strftime('%j'))
    dates = [start - datetime.timedelta(days=x) for x in range(rng)]
    return [x.strftime(DATE_FORMAT) for x in dates]


# Helper functions to gather ando organize Garmin data

Some useful methods from the Garmin class to use
* get_stats - using
* get_steps_data
* get_daily_steps
* get_floors
* get_heart_rates - using
* get_sleep_data - using
* get_stress_data
* get_rhr_day
* get_hrv_data
* get_fitnessage_data - using

Others toook at
* get_activities
* get_activities_fordate
* get_earned_badges

In [106]:
def get_sleep_data(date):

    sleep_data = {}

    to_pop = [
        'id',
        'userProfilePK',
        'napTimeSeconds',
        'sleepWindowConfirmed',
        'sleepWindowConfirmationType',
        'autoSleepStartTimestampGMT',
        'autoSleepEndTimestampGMT',
        'sleepQualityTypePK',
        'sleepResultTypePK',
        'deviceRemCapable',
        'retro',
        'sleepFromDevice',
        'sleepScores',
        'sleepScoreInsight',
        'sleepScorePersonalizedInsight',
        'sleepVersion'
    ]

    data = safe_api_call(api.get_sleep_data, date)[1]

    sleep_data.update(data['dailySleepDTO'])
    if 'sleepScores' in sleep_data:
        sleep_data['sleepScore'] = sleep_data['sleepScores']['overall']['value']
        sleep_data['sleepScoreQuality'] = sleep_data['sleepScores']['overall']['qualifierKey']
        sleep_data['stressSleepQuality'] = sleep_data['sleepScores']['stress']['qualifierKey']
        sleep_data['awakeCountQuality'] = sleep_data['sleepScores']['awakeCount']['qualifierKey']
        sleep_data['remSleepQuality'] = sleep_data['sleepScores']['remPercentage']['qualifierKey']
        sleep_data['restlessnessSleepQuality'] = sleep_data['sleepScores']['restlessness']['qualifierKey']
        sleep_data['lightSleepQuality'] = sleep_data['sleepScores']['lightPercentage']['qualifierKey']
        sleep_data['deepSleepQuality'] = sleep_data['sleepScores']['deepPercentage']['qualifierKey']
    else:
        sleep_data['sleepScore'] = None
        sleep_data['sleepScoreQuality'] = None
        sleep_data['stressSleepQuality'] = None
        sleep_data['awakeCountQuality'] = None
        sleep_data['remSleepQuality'] = None
        sleep_data['restlessnessSleepQuality'] = None
        sleep_data['lightSleepQuality'] = None
        sleep_data['deepSleepQuality'] = None

    # result['sleepHeartRate'] = data['sleepHeartRate']
    sleep_data['avgOvernightHrv'] = data.get('avgOvernightHrv')
    sleep_data['hrvStatus'] = data.get('hrvStatus')
    sleep_data['restingHeartRate'] = data.get('restingHeartRate')

    for key in to_pop:
        try:
            sleep_data.pop(key)
        except KeyError as e:
            pass

    return sleep_data


In [132]:
def get_health_data(date):
    """
    """

    to_pop = [
        'userProfileId',
        'userDailySummaryId',
        'burnedKilocalories',
        'wellnessActiveKilocalories',
        'netRemainingKilocalories',
        'rule',
        'wellnessStartTimeGmt',
        'wellnessStartTimeLocal',
        'wellnessEndTimeGmt',
        'wellnessEndTimeLocal',
        'durationInMilliseconds',
        'wellnessDescription',
        'includesWellnessData',
        'includesActivityData',
        'includesCalorieConsumedData',
        'privacyProtected',
        'floorsAscended',
        'floorsDescended',
        'lastSevenDaysAvgRestingHeartRate',
        'source',
        'lastSyncTimestampGMT',
        'bodyBatteryMostRecentValue',
        'bodyBatteryVersion',
        'averageSpo2',
        'lowestSpo2',
        'latestSpo2',
        'latestSpo2ReadingTimeGmt',
        'latestSpo2ReadingTimeLocal',
        'latestSpo2ReadingTimeLocalaverageMonitoringEnvironmentAltitude',
        'restingCaloriesFromActivity',
        'latestRespirationValue',
        'latestRespirationTimeGMT',
        'respirationAlgorithmVersion',
        'ageGroup',
        'averageMonitoringEnvironmentAltitude',
        'bodyBatteryChargedValue',
        'bodyBatteryDrainedValue',
        'bodyBatteryHighestValue',
        'bodyBatteryLowestValue',
        'bodyBatteryDuringSleep',
        'wellnessKilocalories',
        'consumedKilocalories',
        'remainingKilocalories',
        'netCalorieGoal',
        'wellnessDistanceMeters',
        'userNote',
        'sleepingSeconds',
        'minAvgHeartRate',
        'maxAvgHeartRate',
        'abnormalHeartRateAlertsCount',
        'unmeasurableSleepSeconds',
        'measurableAsleepDuration',
        'measurableAwakeDuration',
        'Stress Percentage',
        'Rest Stress Percentage',
        'Activity Stress Percentage',
        'Uncategorized Stress Percentage',
        'Low Stress Percentage',
        'Medium Stress Percentage',
        'High Stress Percentage',
        'Rest Stress Duration',

    ]

    health_data = safe_api_call(api.get_stats, date)[1]
    health_data['fitnessAge'] = int(safe_api_call(api.get_fitnessage_data, date)[1]['fitnessAge'])

    # Heart rate data
    hdata = safe_api_call(api.get_heart_rates, date)[1]['heartRateValues']
    if hdata:
        heart_rates = [v[1] for v in hdata if v[1] is not None]
        health_data['avgHeartRate'] = float(np.array(heart_rates).mean())

    health_data.update(get_sleep_data(date))

    for key in to_pop:
        try:
            health_data.pop(key)
        except KeyError as e:
            pass

    health_data_clean = {}
    for key, value in health_data.items():
        health_data_clean[lib.convert_camel_case(key).title()] = value

    return health_data_clean.pop('Uuid'), health_data_clean

# Get Health Data

In [133]:
dates = get_date_range()

In [134]:
overall_data = {}
for date in tqdm(dates):
    id, health_data = get_health_data(date)
    overall_data[id] = health_data


100%|██████████| 353/353 [02:31<00:00,  2.32it/s]


# Create the Health Dataframe

In [135]:
df = pd.DataFrame(overall_data).T

In [136]:
df.columns

Index(['Total Kilocalories', 'Active Kilocalories', 'Bmr Kilocalories',
       'Total Steps', 'Total Distance Meters', 'Calendar Date',
       'Daily Step Goal', 'Highly Active Seconds', 'Active Seconds',
       'Sedentary Seconds', 'Moderate Intensity Minutes',
       'Vigorous Intensity Minutes', 'Floors Ascended In Meters',
       'Floors Descended In Meters', 'Intensity Minutes Goal',
       'User Floors Ascended Goal', 'Min Heart Rate', 'Max Heart Rate',
       'Resting Heart Rate', 'Average Stress Level', 'Max Stress Level',
       'Stress Duration', 'Rest Stress Duration', 'Activity Stress Duration',
       'Uncategorized Stress Duration', 'Total Stress Duration',
       'Low Stress Duration', 'Medium Stress Duration', 'High Stress Duration',
       'Stress Percentage', 'Rest Stress Percentage',
       'Activity Stress Percentage', 'Uncategorized Stress Percentage',
       'Low Stress Percentage', 'Medium Stress Percentage',
       'High Stress Percentage', 'Stress Qualifier',
 

In [137]:
df.head()

,Total Kilocalories,Active Kilocalories,Bmr Kilocalories,Total Steps,Total Distance Meters,Calendar Date,Daily Step Goal,Highly Active Seconds,Active Seconds,Sedentary Seconds,...,Sleep Score,Sleep Score Quality,Stress Sleep Quality,Awake Count Quality,Rem Sleep Quality,Restlessness Sleep Quality,Light Sleep Quality,Deep Sleep Quality,Avg Overnight Hrv,Hrv Status
5f3d7033f28b4c96ad99f91a13cdde7e,1282.0,338.0,944.0,4702,3417,2025-12-19,8270,2440,3406,18574,...,88,GOOD,FAIR,EXCELLENT,GOOD,EXCELLENT,EXCELLENT,EXCELLENT,55.0,BALANCED
0249e5d497ac4ce18d6e9896606ee2fa,3154.0,1637.0,1517.0,11132,7929,2025-12-18,7950,3469,17028,42143,...,77,FAIR,FAIR,EXCELLENT,FAIR,EXCELLENT,EXCELLENT,EXCELLENT,54.0,BALANCED
b66ae2c8866943f09ee44c9c702e6464,1791.0,274.0,1517.0,7091,5102,2025-12-17,8160,3280,3964,55636,...,82,GOOD,EXCELLENT,EXCELLENT,GOOD,EXCELLENT,GOOD,EXCELLENT,58.0,BALANCED
3c68feb2861848a6884e16690bed0fff,1927.0,410.0,1517.0,5915,4263,2025-12-16,8400,3154,4381,62785,...,58,POOR,EXCELLENT,EXCELLENT,FAIR,EXCELLENT,EXCELLENT,FAIR,64.0,BALANCED
1ad20af3538c407e8447137312d8d177,3175.0,1658.0,1517.0,14022,10030,2025-12-15,6990,3341,19682,35297,...,69,FAIR,FAIR,FAIR,POOR,FAIR,FAIR,EXCELLENT,47.0,UNBALANCED


# Export the data

In [142]:
df.to_csv(project.raw_file)

# Some Initial queries
Take out 'maxStressLevel'?

In [56]:
agg_columns = [
    'totalKilocalories',
    'activeKilocalories',
    'totalSteps',
    'avgHeartRate',
    'minHeartRate',
    'minAvgHeartRate',
    'maxHeartRate',
    'maxAvgHeartRate',
    'restingHeartRate',
    'averageStressLevel',
    'maxStressLevel',
    'sleepScore',
]

agg_distance = [
    'totalDistanceMeters'
]


agg_seconds = [
    'activeSeconds',
    'sedentarySeconds',
    'sleepingSeconds',
    'sleepTimeSeconds'
]

agg_minutes = [

    'moderateIntensityMinutes',
    'vigorousIntensityMinutes',
]

In [57]:
for item in agg_columns:
    log.info(
        f'{item}'
        f'\n\tMax {item}: {df[item].max():.2f} '
        f'\n\tMin {item}: {df[item].min():.2f} '
        f'\n\tMean {item}: {df[item].mean():.2f}'
    )

PortfolioLogger.Health Overview: INFO: totalKilocalories
	Max totalKilocalories: 4134.00 
	Min totalKilocalories: 1275.00 
	Mean totalKilocalories: 2361.73
PortfolioLogger.Health Overview: INFO: activeKilocalories
	Max activeKilocalories: 2617.00 
	Min activeKilocalories: 0.00 
	Mean activeKilocalories: 856.11
PortfolioLogger.Health Overview: INFO: totalSteps
	Max totalSteps: 36384.00 
	Min totalSteps: 21.00 
	Mean totalSteps: 10761.55
PortfolioLogger.Health Overview: INFO: avgHeartRate
	Max avgHeartRate: 91.67 
	Min avgHeartRate: 64.65 
	Mean avgHeartRate: 75.43
PortfolioLogger.Health Overview: INFO: minHeartRate
	Max minHeartRate: 62.00 
	Min minHeartRate: 32.00 
	Mean minHeartRate: 50.43
PortfolioLogger.Health Overview: INFO: minAvgHeartRate
	Max minAvgHeartRate: 65.00 
	Min minAvgHeartRate: 32.00 
	Mean minAvgHeartRate: 51.57
PortfolioLogger.Health Overview: INFO: maxHeartRate
	Max maxHeartRate: 179.00 
	Min maxHeartRate: 88.00 
	Mean maxHeartRate: 131.77
PortfolioLogger.Health Ove

In [58]:
for item in agg_distance:
    log.info(
        f'{item}'
        f'\n\tMax {item}: {df[item].max():.2f} '
        f'\n\tMin {item}: {df[item].min():.2f} '
        f'\n\tMean {item}: {df[item].mean():.2f}'
    )

PortfolioLogger.Health Overview: INFO: totalDistanceMeters
	Max totalDistanceMeters: 26162.00 
	Min totalDistanceMeters: 15.00 
	Mean totalDistanceMeters: 7745.16


In [59]:
def seconds_to_hour(seconds):
    return seconds / 3600

def minutes_to_hour(minutes):
    return minutes / 60

In [60]:
for item in agg_seconds:
    max = seconds_to_hour(df[item].max())
    min = seconds_to_hour(df[item].min())
    mean = seconds_to_hour(df[item].mean())
    log.info(
        f'{item}'
        f'\n\tMax {item}: {max:.2f} '
        f'\n\tMin {item}: {min:.2f} '
        f'\n\tMean {item}: {mean:.2f}'
    )

PortfolioLogger.Health Overview: INFO: activeSeconds
	Max activeSeconds: 5.47 
	Min activeSeconds: 0.00 
	Mean activeSeconds: 2.27
PortfolioLogger.Health Overview: INFO: sedentarySeconds
	Max sedentarySeconds: 23.39 
	Min sedentarySeconds: 5.04 
	Mean sedentarySeconds: 12.24
PortfolioLogger.Health Overview: INFO: sleepingSeconds
	Max sleepingSeconds: 15.72 
	Min sleepingSeconds: 0.00 
	Mean sleepingSeconds: 7.52
PortfolioLogger.Health Overview: INFO: sleepTimeSeconds
	Max sleepTimeSeconds: 13.57 
	Min sleepTimeSeconds: 3.02 
	Mean sleepTimeSeconds: 7.39
